In [2]:
text = "hello 你好"

tokens = list(text.encode("utf-8"))

print(tokens)
print("字符数:", len(text))
print("字节数:", len(tokens))

[104, 101, 108, 108, 111, 32, 228, 189, 160, 229, 165, 189]
字符数: 8
字节数: 12


In [4]:
#统计所有相邻token Pair出现的次数
def get_stats(ids):
    counts = {}

    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1

    return counts

In [6]:
ids = [1, 2, 1, 2, 1]
stats = get_stats(ids)

print(stats)
print(max(stats, key=stats.get))

{(1, 2): 2, (2, 1): 2}
(1, 2)


In [12]:
#将指定的相邻 Pair 合并成一个新的 Token
def merge(ids, pair, idx):
    new_ids = []
    i = 0
    while i<len(ids):
        if  i<len(ids)-1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            new_ids.append(idx)
            i=i+2
        else:
            new_ids.append(ids[i])
            i=i+1
    return new_ids

In [13]:
ids = [1, 1, 1, 1, 1]

new_ids = merge(ids, (1, 1), 256)

print(new_ids)

[256, 256, 1]


In [15]:
text = "aaabdaaabac"

ids = list(text.encode("utf-8"))
vocab_size = 259

num_merges = vocab_size - 256
merges = {}
vocab = {i: bytes([i]) for i in range(256)}  #推理时用的词表

for i in range(num_merges):
    stats = get_stats(ids)

    pair = max(stats, key=stats.get)
    idx = 256 + i

    ids = merge(ids, pair, idx)

    merges[pair] = idx
    vocab[idx] = vocab[pair[0]] + vocab[pair[1]]

    print(f"Step {i+1}: {pair} -> {idx}")

print("最终 Token IDs:", ids)

Step 1: (97, 97) -> 256
Step 2: (256, 97) -> 257
Step 3: (257, 98) -> 258
最终 Token IDs: [258, 100, 258, 97, 99]


In [16]:
def decode(ids, vocab):
    tokens = b"".join(vocab[idx] for idx in ids)
    return tokens.decode("utf-8")

In [18]:
print(decode([258, 100, 258, 97, 99], vocab))

aaabdaaabac


In [25]:
def encode(text, merges):
    ids = list(text.encode("utf-8"))
    while len(ids)>=2:
        stats = get_stats(ids)
        pair = min(
            stats,
            key=lambda p: merges.get(p, float("inf"))
        )
        if pair not in merges:
            break
        ids = merge(ids, pair, merges.get(pair))
    return ids

In [26]:
encoded = encode("aaabdaaabac", merges)

print(encoded)
print(decode(encoded, vocab))

[258, 100, 258, 97, 99]
aaabdaaabac
